# STB600 – Lab 2: Intensity transformations & spatial filtering

Images: `img.tif` (section 5), `gaussianNoiseImg.tif` and `peppersaltImg.tif` (section 6).
Conda env: `stb600`.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(images, titles, cols=None, size=4):
    """Show grayscale images side by side."""
    cols = cols or len(images)
    rows = int(np.ceil(len(images) / cols))
    plt.figure(figsize=(size * cols, size * rows))
    for i, (im, t) in enumerate(zip(images, titles)):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(im, cmap="gray", vmin=0, vmax=255)
        plt.title(t)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## 5. Intensity Transformations

In [ ]:
img = cv2.imread("img.tif", cv2.IMREAD_GRAYSCALE)
if img is None:
    raise ValueError("img.tif not found next to the notebook.")
print("shape:", img.shape, "dtype:", img.dtype, "min/max:", img.min(), img.max())

# OpenCV window, as in the instructions (press any key to close)
cv2.imshow("Original", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

### 5.1 Image Negative
`s = 255 - r` — bright becomes dark and vice versa.

In [ ]:
neg = cv2.bitwise_not(img)      # same as 255 - img
neg2 = 255 - img
print("bitwise_not == 255 - img:", np.array_equal(neg, neg2))

show([img, neg], ["Original", "Negative"])

**Reflection.** A negative helps when the detail of interest is dark on a large bright area (or bright on dark),
e.g. X-rays, mammograms, biomedical images: the eye distinguishes grey levels better in light tones on dark background.
It adds no information — it only changes how it is presented.

### 5.2 Log Transformation
`s = c · log(1 + r)`, with `c = 255 / log(1 + max(r))` so the output fills 0–255.

In [ ]:
img_float = img.astype(np.float32)

c = 255 / np.log(1 + np.max(img_float))
log_img = c * np.log1p(img_float)
log_img = cv2.normalize(log_img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

show([img, log_img], ["Original", "Log transform"])

# transfer curve
r = np.arange(256)
plt.plot(r, c * np.log1p(r)); plt.plot(r, r, "--")
plt.xlabel("input r"); plt.ylabel("output s"); plt.title("Log transform curve"); plt.show()

**Effect.** Dark values are stretched (steep curve near 0), bright values compressed. Good for images with a large
dynamic range (e.g. Fourier spectra) where details hide in the dark. Limitation: bright regions lose contrast and
there is no parameter to tune the strength (only `c`).

### 5.3 Gamma (Power-Law) Transformation
`s = c · r^γ` on normalised `r ∈ [0, 1]`. γ < 1 brightens dark regions, γ > 1 darkens / enhances bright regions.

In [ ]:
def gamma_transform(img, gamma, c=1.0):
    norm = img.astype(np.float32) / 255.0
    out = c * np.power(norm, gamma)
    return np.clip(out * 255, 0, 255).astype(np.uint8)

gammas = [0.3, 0.5, 1.0, 1.5, 3.0]
show([gamma_transform(img, g) for g in gammas], [f"gamma = {g}" for g in gammas], cols=5, size=3.5)

r = np.linspace(0, 1, 256)
for g in gammas:
    plt.plot(r * 255, (r ** g) * 255, label=f"γ={g}")
plt.xlabel("input r"); plt.ylabel("output s"); plt.legend(); plt.title("Gamma curves"); plt.show()

**Effect.** Like log, but tunable: γ < 1 expands dark tones, γ > 1 expands bright tones (e.g. washed-out images).
Used for display/monitor gamma correction. Limitation: one global curve for the whole image — it cannot enhance
dark and bright regions at the same time, and extreme γ clips detail at one end.

## 6. Spatial Filtering
Using `gaussianNoiseImg.tif` and `peppersaltImg.tif`. Kernel sizes 3, 5, 9 for comparison.

In [ ]:
gauss = cv2.imread("gaussianNoiseImg.tif", 0)
sp = cv2.imread("peppersaltImg.tif", 0)
for name, im in [("gaussianNoiseImg", gauss), ("peppersaltImg", sp)]:
    if im is None:
        raise ValueError(f"{name}.tif not found")
    print(name, im.shape, im.dtype)

show([gauss, sp], ["Gaussian noise", "Salt & pepper noise"])
ksizes = [3, 5, 9]

### 6.1 Averaging Filter

In [ ]:
for name, im in [("Gaussian noise", gauss), ("Salt & pepper", sp)]:
    outs = [cv2.blur(im, (k, k)) for k in ksizes]
    show([im] + outs, [name] + [f"blur {k}x{k}" for k in ksizes], size=3.5)

# same thing with filter2D and a uniform kernel
k = 5
kernel = np.ones((k, k), np.float32) / (k * k)
f2d = cv2.filter2D(gauss, -1, kernel)
print("max |blur - filter2D|:", np.abs(f2d.astype(int) - cv2.blur(gauss, (k, k)).astype(int)).max())

### 6.2 Gaussian Filter

In [ ]:
for name, im in [("Gaussian noise", gauss), ("Salt & pepper", sp)]:
    outs = [cv2.GaussianBlur(im, (k, k), 0) for k in ksizes]   # sigma=0 -> computed from kernel size
    show([im] + outs, [name] + [f"Gaussian {k}x{k}" for k in ksizes], size=3.5)

### 6.3 Median Filter

In [ ]:
for name, im in [("Gaussian noise", gauss), ("Salt & pepper", sp)]:
    outs = [cv2.medianBlur(im, k) for k in ksizes]
    show([im] + outs, [name] + [f"median {k}x{k}" for k in ksizes], size=3.5)

### Comparison (5x5)

In [ ]:
k = 5
for name, im in [("Gaussian noise", gauss), ("Salt & pepper", sp)]:
    show([im, cv2.blur(im, (k, k)), cv2.GaussianBlur(im, (k, k), 0), cv2.medianBlur(im, k)],
         [name, "Averaging", "Gaussian", "Median"], size=3.5)

### Reflection notes
- **Best on salt & pepper: median.** The noise pixels are outliers (0 or 255); the median ignores them, while
  mean/Gaussian average them into grey blotches. Median also keeps edges sharp because it outputs an existing value.
- **Bigger kernel** → stronger smoothing / less noise, but more blur and loss of fine detail and edges
  (median with large kernels starts to remove thin lines and round corners).
- **Averaging vs Gaussian:** averaging weights all neighbours equally → stronger, boxy blur. Gaussian weights the
  centre more → smoother, more natural blur that preserves edges a bit better at the same kernel size. Both work
  reasonably on Gaussian noise; neither is good on salt & pepper.